# Managing short-term memory: deleting messages

Trimming only hides old messages from the LLM; they stay saved. **Deleting** removes them from the
state (and so from the checkpointer) for good, by returning `RemoveMessage(id=...)` from a node.

```
START -> chat_node -> delete_old_messages -> END
```

In [1]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, RemoveMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

model = ChatGroq(model='openai/gpt-oss-20b', reasoning_effort='low')

In [2]:
KEEP_LAST = 4  # keep the last 4 messages = the last 2 question/answer pairs


def chat_node(state: MessagesState):
    return {'messages': [model.invoke(state['messages'])]}


def delete_old_messages(state: MessagesState):
    # add_messages sees a RemoveMessage and deletes the message with that id from the state
    old = state['messages'][:-KEEP_LAST]
    return {'messages': [RemoveMessage(id=m.id) for m in old]}


graph = StateGraph(MessagesState)
graph.add_node('chat_node', chat_node)
graph.add_node('delete_old_messages', delete_old_messages)
graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', 'delete_old_messages')
graph.add_edge('delete_old_messages', END)

chatbot = graph.compile(checkpointer=InMemorySaver())
config = {'configurable': {'thread_id': 'delete-1'}}

In [3]:
def chat(text):
    result = chatbot.invoke({'messages': [HumanMessage(content=text)]}, config=config)
    print(f'You: {text}\nAI:  {result["messages"][-1].content[:110]}')
    print(f'     (stored {len(result["messages"])} messages)\n')


chat('Hi! My name is Priyansh.')
chat('What is my name?')
chat('Give me one short fact about Paris.')
chat('Give me one short fact about Tokyo.')
chat('What is my name?')  # the intro was deleted, so the name is gone for good

You: Hi! My name is Priyansh.
AI:  Hello Priyansh! 👋 How can I help you today?
     (stored 2 messages)



You: What is my name?
AI:  Your name is Priyansh.
     (stored 4 messages)



You: Give me one short fact about Paris.
AI:  Paris is home to the world’s most visited art museum, the Louvre, which houses over 35,000 works, including th
     (stored 4 messages)



You: Give me one short fact about Tokyo.
AI:  Tokyo is the world’s largest metropolitan economy, with a GDP that rivals many countries.
     (stored 4 messages)



You: What is my name?
AI:  I don’t have that information. If you’d like me to address you by name, just let me know what you’d like me to
     (stored 4 messages)



In [4]:
# What's really saved now: only the last KEEP_LAST messages
for m in chatbot.get_state(config).values['messages']:
    print(f'{m.type:5} {m.content[:70]}')

human Give me one short fact about Tokyo.
ai    Tokyo is the world’s largest metropolitan economy, with a GDP that riv
human What is my name?
ai    I don’t have that information. If you’d like me to address you by name


In [5]:
# Clear the whole conversation (e.g. a 'Clear chat' button): update the state from outside the graph
chatbot.update_state(config, {'messages': [RemoveMessage(id=REMOVE_ALL_MESSAGES)]})
print('messages after clearing:', len(chatbot.get_state(config).values['messages']))

messages after clearing: 0
